# Synthetic call-transcript generator — Azure `o4-mini`

**What this is (honestly):** an automation of the method that built the shipped
synthetic corpus. Those transcripts were **generated by an LLM** This
notebook runs the **same loop** with Azure `o4-mini` so it is reproducible
standalone: an LLM drafts a realistic, messy transcript per the spec, then the exact
gold-integrity checker used originally (`compute_standalone_counts`) verifies the
labels before saving.

The spec is the real one:
- **`sg_pii_reference_bank.md`** — fake SG seed values (NRIC / phone / A–Z addresses),
  loaded and injected into the prompt.
- **Messiness rules** — never insert a value as a clean string; spell it out, interrupt
  it, restate it in another format, add STT-phonetic misheard forms.
- **Conventions** — NRIC usually only the last 4 chars; addresses tagged
  compositionally (street / block / unit / postal separately).
- **Gold invariant** — every surface string listed once per real occurrence.

### Prerequisites
- `pip install -r requirements.txt` (installs `langchain-openai`).
- The standard company `AZURE_OPENAI_*` environment variables.

### Workflow (staging)
Generate → review `data/generated/` (fix/drop `_REVIEW` files) → move approved ones
into `data/train/synthetic/` (or `data/val/synthetic/`) → `build_splits.py`.

**Data safety:** ships output-free; the seed bank is fake; `data/generated/` is gitignored.

## 0. Config + load the reference bank

In [ ]:
import re, json
from pathlib import Path
from collections import Counter

HERE = Path.cwd()
while HERE.name and not (HERE / 'inference' / 'labels.py').exists():
    HERE = HERE.parent                       # find the pii/ root
DATA_PREP = HERE / 'finetuning' / 'data_prep'
OUT_DIR = HERE / 'data' / 'generated'        # staging folder (gitignored)

# The fake seed values the corpus was drawn from — injected into the prompt.
REF_BANK = (DATA_PREP / 'sg_pii_reference_bank.md').read_text(encoding='utf-8')

START_INDEX = 1000                           # filenames continue NNN_tier_scenario.json
LABELS = ['sg_phone_number', 'sg_nric_fin', 'sg_address', 'sg_postal_code',
          'sg_address_unit_number', 'sg_address_block_number', 'email_address',
          'account_number', 'full_name']
print(f'reference bank: {len(REF_BANK.splitlines())} lines; output -> {OUT_DIR}')

## 1. Azure `o4-mini` client

Same client as the leak-judge notebook — the company Azure deployment via the
standard `AZURE_OPENAI_*` environment variables.

In [ ]:
from langchain_openai import AzureChatOpenAI

def get_o4_mini():
    """Azure o4-mini. Expects the standard AZURE_OPENAI_* environment variables."""
    return AzureChatOpenAI(
        azure_deployment='o4-mini',
        api_version='2024-12-01-preview',
        model_name='o4-mini',
        max_completion_tokens=16000,
        timeout=300,
    )

## 2. The generation prompt (the real spec)

The tier controls **how messy the PII delivery is**. The prompt injects the
reference bank and enforces every rule the original authoring followed: draw fake
values from the bank, run them through real-transcription messiness, keep the NRIC
last-4 convention, tag addresses compositionally, and obey the gold invariant.

| tier | delivery |
|---|---|
| `normal` | PII stated fairly plainly; at most one read-back |
| `medium` | some spoken-word numbers / small groups; one or two read-backs |
| `hard` | digit-by-digit, split across turns, corrections, email spelled out, several read-backs, sometimes two people |
| `negative` | a real call with **no PII at all** — teaches the model not to over-redact (entities `{}`) |

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers.json import JsonOutputParser

TIER_GUIDE = {
    'normal':   'PII stated fairly plainly and clearly. At most one agent read-back.',
    'medium':   'Some numbers spoken as words or small groups; the agent reads one or two values back, so those appear twice.',
    'hard':     'Very messy: digits one-by-one, values split across turns, corrections/restarts, an email spelled letter-by-letter, MULTIPLE read-backs, sometimes two people (e.g. account holder + joint holder) each with their own NRIC/phone.',
    'negative': 'A realistic call with NO personal data at all (billing FAQ, appliance energy question, opening hours). The entities dict MUST be empty.',
}

GEN_PROMPT = PromptTemplate.from_template(
"""You are generating ONE realistic Singapore SP Group (electricity/utilities)
call-centre transcript to train a PII-redaction model. Write it like a real, messy,
disfluent phone call transcribed by speech-to-text -- NOT clean template text.

Scenario: {scenario}
Difficulty tier: {tier} -- {tier_guide}

=== PII VALUES: draw from this reference bank of FAKE seed values ===
{ref_bank}

Rules for using the bank:
- Pick values FROM the bank, and VARY them: for this transcript prefer addresses whose
  road name starts with one of these letters: {focus_letters}. Use different NRIC / phone
  values than the first listed examples.
- NEVER insert a value as a clean string. Always run it through real-transcription
  messiness: spelled out digit-by-digit, interrupted across turns, restated in a different
  format, and STT-phonetic misheard forms (e.g. 's one two three ... dog' for S...D).
- NRIC convention: in real calls the customer usually gives ONLY THE LAST 4 CHARACTERS
  (e.g. '567D') after the agent asks 'last four characters of your NRIC'. Prefer that; a
  full 9-character NRIC is rare.
- Addresses are COMPOSITIONAL -- tag the parts separately: the street name as sg_address,
  the block as sg_address_block_number, the unit as sg_address_unit_number, the 6-digit
  postal code as sg_postal_code.

Format:
- Speaker-labelled, alternating: SPEAKER_00 = agent, SPEAKER_01 = customer.
- LONG and realistic: roughly 3200-4200 characters, 18-28 turns, with natural small talk,
  hold/verification moments and clarifications. Do NOT produce a short, clipped call.

Label set (use ONLY these keys, omit any unused):
  sg_phone_number, sg_nric_fin, sg_address, sg_postal_code, sg_address_unit_number,
  sg_address_block_number, email_address, account_number, full_name

GOLD INVARIANT (critical): in 'entities', list each surface string ONCE PER OCCURRENCE in
the transcript. Count real occurrences -- if the customer spells a number AND the agent
reads it back, both surface forms appear, so list both ('nine one two...' and '9123'); if a
value appears twice, list it twice. Every listed value must appear VERBATIM in the text.

Return ONLY this JSON, no prose:
{{"transcript": "<full speaker-labelled transcript, \\n between turns>",
  "entities": {{"<label>": ["<exact surface string>", ...]}}}}
""")

gen_chain = GEN_PROMPT | get_o4_mini() | JsonOutputParser()

## 3. Gold-integrity checker (the exact one used to build the corpus)

`compute_standalone_counts` counts each value's occurrences with word boundaries and
**longest-value-first span exclusion**, so a short value (bare block `45`) is never
double-counted inside a longer one it sits in (`610045`, `Block 45`). The check compares
the *intended* count (how many times a value is listed) against that *standalone* count
— the exact under-count guard from the original `generate_batch_*.py` scripts.

In [ ]:
def core_format(s):
    return s.rstrip('.,?!- ')

def compute_standalone_counts(values, text):
    """{value: standalone occurrence count}; longest-first, covered-span exclusion."""
    distinct = sorted(set(values), key=len, reverse=True)
    covered, counts = [], {}
    for v in distinct:
        core = core_format(v)
        if not core:
            counts[v] = 0; continue
        pat = re.compile(r'(?<![A-Za-z0-9])' + re.escape(core) + r'(?![A-Za-z0-9])')
        spans = []
        for m in pat.finditer(text):
            ms, me = m.start(), m.end()
            if any(cs <= ms and me <= ce for cs, ce in covered):
                continue
            spans.append((ms, me))
        counts[v] = len(spans); covered.extend(spans)
    return counts

def check_gold_invariant(text, entities):
    """Return a list of problems; empty list == clean."""
    problems = []
    unknown = set(entities) - set(LABELS)
    if unknown:
        problems.append(f'unknown label key(s): {sorted(unknown)}')
    intended = Counter(v for vals in entities.values() for v in vals)
    actual = compute_standalone_counts(list(intended), text)
    for v, n in intended.items():
        if actual.get(v, 0) != n:
            problems.append(f'{v!r}: listed {n}x but occurs {actual.get(v, 0)}x (standalone) in transcript')
    return problems

## 4. Generate one transcript (draft → verify → repair)

Draft with o4-mini → run the gold checker → if it complains, show the model the
problems and ask it to fix ONLY the counts → re-check. Anything still failing is saved
with a `_REVIEW` marker for a human, exactly like the original manual fix step.

In [ ]:
ALPHABET = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

def _slug(scenario):
    return re.sub(r'[^a-z0-9]+', '_', scenario.lower()).strip('_')[:48]

def _focus_letters(index):
    # rotate through the alphabet so successive transcripts favour different addresses
    return ', '.join(ALPHABET[(index + k) % 26] for k in (0, 9, 18))

REPAIR_PROMPT = PromptTemplate.from_template(
    'A checker found these gold-count problems in a transcript:\n{problems}\n\n'
    'Fix ONLY the entities dict so each value is listed exactly as many times as it occurs '
    'in the transcript (the gold invariant). Do not change the transcript.\n\n'
    'transcript:\n{transcript}\n\ncurrent entities:\n{entities}\n\n'
    'Return ONLY corrected JSON: {{"transcript": <unchanged>, "entities": {{...}}}}')
repair_chain = REPAIR_PROMPT | get_o4_mini() | JsonOutputParser()

def generate_one(scenario, tier, index, save=True):
    out = gen_chain.invoke({'scenario': scenario, 'tier': tier,
                            'tier_guide': TIER_GUIDE[tier], 'ref_bank': REF_BANK,
                            'focus_letters': _focus_letters(index)})
    text, ents = out['transcript'], out.get('entities', {})
    problems = check_gold_invariant(text, ents)
    if problems:
        fixed = repair_chain.invoke({'problems': '\n'.join(problems),
                                     'transcript': text,
                                     'entities': json.dumps(ents, ensure_ascii=False)})
        ents = fixed.get('entities', ents)
        problems = check_gold_invariant(text, ents)
    marker = '' if not problems else '_REVIEW'
    fname = f'{index:03d}_{tier}_{_slug(scenario)}{marker}.json'
    if save:
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        (OUT_DIR / fname).write_text(
            json.dumps({'input': text, 'output': {'entities': ents}}, ensure_ascii=False),
            encoding='utf-8')
    status = 'OK' if not problems else 'NEEDS REVIEW: ' + '; '.join(problems)
    print(f'{fname:60} {len(text):5d} chars  {sum(len(v) for v in ents.values()):3d} gold  [{status}]')
    return fname, {'input': text, 'output': {'entities': ents}}, problems

## 5. Batch generation

Edit `SCENARIOS` (`(scenario, tier)` pairs) to whatever mix you want. A healthy batch
spreads tiers and includes some `negative` calls. Output goes to `data/generated/` for
review — it is NOT training data until you move approved files into `data/train/synthetic/`.

In [ ]:
SCENARIOS = [
    ('customer updates mailing address after moving', 'normal'),
    ('customer updates contact number, agent reads it back', 'medium'),
    ('verify identity with last 4 of NRIC before a billing change', 'medium'),
    ('NRIC update after a legal name change, joint account holder also mentioned', 'hard'),
    ('online account registration, email spelled out letter by letter', 'hard'),
    ('general question about ceiling fan vs aircon electricity usage', 'negative'),
]

results = []
for i, (scenario, tier) in enumerate(SCENARIOS):
    results.append(generate_one(scenario, tier, START_INDEX + i))

review = [f for f, _r, p in results if p]
print(f'\n{len(results)} generated in {OUT_DIR}, {len(review)} need review: {review}')

## 6. Verify the staging folder

Re-scan `data/generated/` and run the gold-integrity check across everything, so you
know what is clean before moving files into the training split.

In [ ]:
clean = flagged = 0
for p in sorted(OUT_DIR.glob('*.json')):
    rec = json.loads(p.read_text(encoding='utf-8'))
    probs = check_gold_invariant(rec['input'], rec['output']['entities'])
    if probs:
        flagged += 1; print(f'FLAGGED {p.name}: {probs}')
    else:
        clean += 1
print(f'\n{clean} clean, {flagged} flagged in {OUT_DIR}')
print('Next: fix/drop flagged files, move approved into data/train/synthetic/, run build_splits.py')